# SenseVoice 학습 노트북

이 노트북은 SenseVoice (FunASR) 모델을 한국어 GOLD JSONL 로 파인튜닝하기 위한 셀 골격입니다.

## 사용 흐름

1. 환경 점검 (셀 1~3)
2. 데이터 로드 + 검증 (셀 4~5)
3. SenseVoice 형식 어댑터 (셀 6)
4. 학습 명령 빌드 + 실행 (셀 7~8) — `torchrun` 외부 프로세스
5. 평가 (셀 9)

## 약속

- 핵심 로직은 `project/` 모듈에. 노트북은 *호출만*.
- 모든 경로/백본/W&B 는 `configs/<exp>.yaml` 에서 읽는다. 노트북에 박지 않는다.

## 셀 1 — 환경 점검

In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO = Path('/home/cssong/workspace/TRAIN-ASR')
sys.path.insert(0, str(REPO))
os.chdir(REPO)

# conda 환경 / GPU 빠른 확인
print('CONDA_DEFAULT_ENV:', os.environ.get('CONDA_DEFAULT_ENV'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv'], check=False)

## 셀 2 — Config 로드

In [ ]:
from project.utils.config import load_config
from project.utils.seed import seed_everything

CFG_PATH = REPO / 'configs/default.yaml'   # 새 실험은 자신의 yaml 만들기
cfg = load_config(CFG_PATH)
seed_everything(cfg['experiment']['seed'])
print('experiment:', cfg['experiment']['name'])
print('backbone:  ', cfg['models']['sensevoice']['backbone'])
print('train data:', cfg['paths']['train_jsonl'])

## 셀 3 — 의존성 sanity

In [ ]:
import importlib

needed = ['funasr', 'jiwer', 'soundfile', 'torch']
for pkg in needed:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'  {pkg:12s} {v}')
    except ImportError:
        print(f'  {pkg:12s} ❌ 미설치')

import torch
print('CUDA available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())

GOLD JSONL 로드 + 자동 검증. 실패 시 `SchemaValidationError`.

In [ ]:
from project.data import load_samples

train = load_samples(cfg['paths']['train_jsonl'])
val   = load_samples(cfg['paths']['val_jsonl'])
print(f'train: {len(train):,}  /  val: {len(val):,}')
print('sample[0]:', train[0])

## 셀 5 — 화자 분포 / 도메인 분포 빠른 점검

In [ ]:
from collections import Counter

print(f'화자 수: {len({s.speaker_id for s in train}):,}')
print(f'코퍼스: {Counter(s.corpus_id for s in train).most_common()}')
print(f'gender: {Counter(s.gender for s in train).most_common()}')
print(f'age:    {Counter(s.age_group for s in train).most_common()}')

## 셀 6 — SenseVoice JSONL 어댑터 (GOLD → FunASR 입력)

어댑터 로직은 `project/data/adapters/sensevoice.py` 에 둘 것 (구현 예정).
여기선 *호출만*.

In [ ]:
# TODO: 어댑터 구현 후 활성화
# from project.data.adapters.sensevoice import to_sensevoice_jsonl
#
# sv_train = REPO / 'data/GOLD/adapted/sensevoice/train.jsonl'
# sv_val   = REPO / 'data/GOLD/adapted/sensevoice/val.jsonl'
# n = to_sensevoice_jsonl(train, sv_train); print(f'wrote {n} → {sv_train}')
# n = to_sensevoice_jsonl(val,   sv_val);   print(f'wrote {n} → {sv_val}')

print('어댑터 구현 후 활성화 — project/data/adapters/sensevoice.py')

## 셀 7 — 학습 명령 빌드 (`torchrun` 셸 명령 출력)

FunASR 의 `train_ds.py` 는 `torchrun` 으로 분산 학습 — 노트북 셀에서 직접 실행보다
별도 터미널에서 `bash scripts/finetune_sensevoice.sh` 권장.

여기선 명령만 출력 → 복사하여 외부 터미널 실행.

In [ ]:
import funasr

train_tool = Path(funasr.__path__[0]) / 'bin' / 'train_ds.py'
gpus = cfg['runtime']['cuda_visible_devices']
gpu_num = len(gpus.split(','))
output_dir = Path(cfg['paths']['outputs_dir']) / cfg['experiment']['name']

cmd = f'''
export CUDA_VISIBLE_DEVICES="{gpus}"
torchrun --nnodes 1 --nproc_per_node {gpu_num} --master_port 26669 {train_tool} \\
    ++model="{cfg['models']['sensevoice']['backbone']}" \\
    ++trust_remote_code=true \\
    ++train_data_set_list="{cfg['paths']['train_jsonl']}" \\
    ++valid_data_set_list="{cfg['paths']['val_jsonl']}" \\
    ++dataset_conf.batch_size={cfg['training']['batch_size'] * 1000} \\
    ++dataset_conf.batch_type="token" \\
    ++train_conf.max_epoch={cfg['training']['epochs']} \\
    ++train_conf.log_interval=1 \\
    ++train_conf.validate_interval=2000 \\
    ++train_conf.save_checkpoint_interval=2000 \\
    ++optim_conf.lr={cfg['training']['lr']} \\
    ++output_dir="{output_dir}"
'''.strip()

print(cmd)
print()
print(f'→ 위 명령을 별도 터미널에서 실행. 로그는 {output_dir}/train.log')

## 셀 8 — (선택) subprocess 로 백그라운드 실행 + 로그 tail

In [ ]:
# 신중하게 사용 — 노트북 커널이 꺼지면 학습도 죽을 수 있음.
# 가급적 tmux/screen + bash 권장.
#
# import subprocess
# log = open(output_dir / 'train.log', 'w')
# proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
# print(f'started PID={proc.pid}, log={log.name}')

## 셀 9 — 학습 후 평가

학습 산출물을 평가 모듈에 넘기면 `evaluation_report.{txt,json}` + `<run>_diff.txt` 가 생성됩니다.

In [ ]:
# TODO: 평가 모듈 구현 후 활성화
# from project.evaluation import evaluate_on_benchmark_suite
#
# report = evaluate_on_benchmark_suite(
#     model_path=output_dir,                 # 또는 model.pt.best
#     model_type='sensevoice',
#     benchmarks=cfg['eval']['benchmarks'],  # 'all' or [...]
#     out_dir=Path(cfg['paths']['eval_results_dir']) / cfg['experiment']['name'],
# )
# print(report)

print('평가 모듈 구현 후 활성화 — project/evaluation/')